# Attendance System Colab Setup
This notebook sets up and runs the face recognition attendance system on Google Colab.

In [ ]:
# Cell 1 — Install dependencies
!pip install face_recognition dlib opencv-python-headless numpy pandas matplotlib

In [ ]:
# Cell 2 — Upload files
import os
from google.colab import files

os.makedirs('data', exist_ok=True)
os.makedirs('test-video', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("Upload encodings.pkl")
uploaded_enc = files.upload()
for filename in uploaded_enc.keys():
    os.rename(filename, os.path.join('data', 'encodings.pkl'))
    print(f"Moved {filename} to data/encodings.pkl")

print("\nUpload the test video file")
uploaded_vid = files.upload()
global_video_path = ""
for filename in uploaded_vid.keys():
    global_video_path = os.path.join('test-video', filename)
    os.rename(filename, global_video_path)
    print(f"Moved {filename} to {global_video_path}")

In [ ]:
# Cell 3 — Copy project files
from google.colab import files
print("Upload recognize.py, webhook_logger.py, main.py")
uploaded_scripts = files.upload()

In [ ]:
# Cell 4 — Patch video source
import re

with open('recognize.py', 'r') as f:
    content = f.read()

# Replace the hardcoded Windows path with the uploaded video path
# Using a regex to find the video_source assignment in run_recognition
pattern = r'video_source\s*=\s*r"F:\\Zain\\Konversation\\dev-mvp-face-detection\\attendance-system\\test-video\\provided2\.mp4"'
replacement = f'video_source = "{global_video_path}"'
new_content = re.sub(pattern, replacement, content)

with open('recognize.py', 'w') as f:
    f.write(new_content)

print(f"Patched recognize.py with video path: {global_video_path}")

In [ ]:
# Cell 5 — Process video and save output
import cv2
import os
import datetime
import recognize
import webhook_logger

attendance_log = []

def on_identify(name, guardian_no, guardian_name, school_name):
    if name == "Unknown":
        return
    if webhook_logger.log_attendance(name, guardian_no, guardian_name, school_name):
        print(f"✓ {name} marked present")
        attendance_log.append({
            "student_name": name,
            "guardian_name": guardian_name,
            "guardian_no": guardian_no,
            "school_name": school_name,
            "arrival_time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

def run_recognition_colab_video(on_identify=None, video_source=None):
    known_encodings, known_names, known_guardian_nos, known_guardian_names, known_school_names = recognize.load_encodings()
    
    if video_source is None:
        video_source = global_video_path

    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        print(f"Error: could not open video source: {video_source}")
        return

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    os.makedirs('output', exist_ok=True)
    output_path = 'output/attended_output.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    total_detections = 0
    tracker = {}
    announced = set()
    current_faces = []

    print(f"Total frames to process: {total_frames}")
    print(f"Processing {video_source}... writing to {output_path}")
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1

            if frame_count % recognize.FRAME_SKIP == 0:
                current_faces = recognize.identify_face(frame, known_encodings, known_names, known_guardian_nos, known_guardian_names, known_school_names)
                total_detections += len(current_faces)

                for face in current_faces:
                    name = face[4]
                    if name == "Unknown":
                        continue
                    if name not in tracker:
                        tracker[name] = {
                            "count": 0,
                            "persistence": recognize.PERSISTENCE_LIMIT,
                            "guardian_no": face[6],
                            "guardian_name": face[7],
                            "school_name": face[8]
                        }
                    tracker[name]["count"] += 1
                    tracker[name]["persistence"] = recognize.PERSISTENCE_LIMIT

                    if tracker[name]["count"] >= recognize.CONFIRMATION_THRESHOLD and name not in announced:
                        announced.add(name)
                        if on_identify:
                            on_identify(name, tracker[name]["guardian_no"], tracker[name]["guardian_name"], tracker[name]["school_name"])

            recognize.draw_results(frame, current_faces, tracker)
            out.write(frame)

            if frame_count % 50 == 0:
                print(f"Progress: {frame_count}/{total_frames} frames processed...")

    finally:
        cap.release()
        out.release()
    
    print(f"Finished. Output saved to {output_path}")
    print(f"Total faces detected across video: {total_detections}")
    
run_recognition_colab_video(on_identify=on_identify, video_source=global_video_path)

In [ ]:
# Cell 6 — Play video inline in Colab
!ffmpeg -y -i output/attended_output.mp4 -vcodec libx264 output/attended_h264.mp4 -loglevel quiet

from IPython.display import HTML
from base64 import b64encode
mp4 = open('output/attended_h264.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=640 controls autoplay><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
# Cell 7 — Show session summary
import pandas as pd

if attendance_log:
    df = pd.DataFrame(attendance_log)
    print("Attendance recorded this session:")
    display(df)
else:
    print("No attendance recorded this session.")